In [26]:
import pandas as pd
import numpy as np

1. LOAD RAW DATA

In [ ]:
df = pd.read_csv("netflix_titles_RAW.csv")

print("Original shape:", df.shape)
print("\nOriginal columns:")
print(df.columns.tolist())

Original shape: (8807, 12)

Original columns:
['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']


2. STANDARDIZE COLUMN HEADERS
     - Remove leading/trailing  spaces
     - Convert to lowercase
     - Replace spaces with underscores

In [28]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("\nCleaned column names:")
print(df.columns.tolist())



Cleaned column names:
['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']


3. HANDLE MISSING VALUES

In [29]:
print("\nMissing values before cleaning:")
print(df.isnull().sum())


Missing values before cleaning:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64


In [30]:
# Replace missing values in text/categorical columns
text_columns = [
    "director",
    "cast",
    "country",
    "date_added",
    "duration",
    "rating"
]

for col in text_columns:
    df[col] = df[col].fillna("Unknown")

In [31]:
print("\nMissing values after cleaning:")
print(df.isnull().sum())


Missing values after cleaning:
show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64


4. FIX INCORRECT VALUES IN RATING / DURATION

In this dataset, three movie durations were incorrectly
stored in the rating column:


*   74 min
*   84 min
*   66 min






In [32]:
# Move these values to duration.

incorrect_duration_values = ["66 min", "74 min", "84 min"]

mask = df["rating"].isin(incorrect_duration_values)

df.loc[mask, "duration"] = df.loc[mask, "rating"]

# Set rating to missing for these records
df.loc[mask, "rating"] = np.nan

# Fill the corrected missing rating values
df["rating"] = df["rating"].fillna("Unknown")

5. STANDARDIZE TEXT VALUES

In [33]:
# Remove leading/trailing spaces from all text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# Remove multiple spaces inside text
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.replace(r"\s+", " ", regex=True)


# Standardize "type"
df["type"] = df["type"].str.strip().str.title()

# Standardize country names
df["country"] = df["country"].str.strip()

# Standardize rating
df["rating"] = df["rating"].str.strip()

# Standardize show_id
df["show_id"] = df["show_id"].str.strip()

7. CHECK AND FIX DATA TYPES

In [34]:
# show_id should be text
df["show_id"] = df["show_id"].astype("string")

# type should be text
df["type"] = df["type"].astype("string")

# title should be text
df["title"] = df["title"].astype("string")

# release_year should be integer
df["release_year"] = pd.to_numeric(
    df["release_year"],
    errors="coerce"
).astype("Int64")

# rating should be text
df["rating"] = df["rating"].astype("string")

# duration should be text because it contains values such as:
# "90 min", "2 Seasons", "1 Season"
df["duration"] = df["duration"].astype("string")

10. REMOVE DUPLICATE ROWS

In [36]:
print("\nDuplicate rows before removal:")
print(df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicate rows after removal:")
print(df.duplicated().sum())



Duplicate rows before removal:
0
Duplicate rows after removal:
0


12. FINAL DATA QUALITY CHECK

In [38]:
print("\nFinal shape:", df.shape)

print("\nFinal missing values:")
print(df.isnull().sum())

print("\nFinal data types:")
print(df.dtypes)

print("\nFirst 5 rows:")
print(df.head())


Final shape: (8807, 12)

Final missing values:
show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64

Final data types:
show_id         string[python]
type            string[python]
title           string[python]
director                object
cast                    object
country                 object
date_added              object
release_year             Int64
rating          string[python]
duration        string[python]
listed_in               object
description             object
dtype: object

First 5 rows:
  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  Tv Show          Blood & Water          Unknown   
2      s3  Tv Show              Ganglands  Julien Leclercq   
3      s4  Tv Show  Jailbirds New Orleans          Unkn

13. SAVE CLEANED DATA

In [39]:
df.to_csv(
    "netflix_titles_CLEANED.csv",
    index=False
)

print("\nCleaning completed successfully!")
print("Saved as: netflix_titles_CLEANED.csv")


Cleaning completed successfully!
Saved as: netflix_titles_CLEANED.csv
